# Post 1 — When an xG "Justice Table" Mistakes Luck for Finishing

**Analysis + figures only.** No article prose here; the writeup is authored separately from these
outputs.

The goals−xG table is not arithmetically wrong, but it is often *mislabelled*. It is a **scoreboard**
residual (all goals − xG) frequently read as a **finishing** residual. This notebook decomposes the
two, side by side:

- **Scoreboard residual** = (all goals) − xG  → includes fortune like own goals.
- **Finishing residual** = (non-own goals) − xG  → the part a team's own attempts actually drove.

Data provenance: results/goals from **openfootball** (public domain); xG & team match stats from the
**Kaggle FIFA World Cup 2026** dataset (CC0). Join key is our `match_id`.

> Charts are **Plotly** (for later Quarto embedding). Needs the `viz` extra:
> `uv sync --extra viz` then run with that environment's kernel.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import poisson

from fsc.utils import paths

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", None)


def load(name: str) -> pd.DataFrame:
    """Read a processed parquet table by stem name (no hardcoded paths)."""
    return pd.read_parquet(paths.PROCESSED / f"{name}.parquet")


matches = load("matches")
goals = load("goals")
team_stats = load("team_match_stats")

played_ids = set(matches.loc[matches["status"] == "played", "match_id"])

# Snapshot stamp read from the data itself (not hardcoded).
_snap = pd.to_datetime(matches["snapshot_date"]).max().date()
STAMP = f"FIFA World Cup 2026 — matches completed through {_snap:%B %-d, %Y}."
print("played matches:", len(played_ids))
print("stamp:", STAMP)

## Task 0 — Own-goal xG audit (do this before any correction)

There are 14 own goals in `goals.parquet` (`owngoal == True`). Before "removing" any of them we must
know how xG relates to an own goal, because that decides what the removal actually does to the
finishing residual.

In [ ]:
# What xG granularity does this dataset actually have?
match_events = pd.read_csv(paths.RAW / "kaggle_wc2026" / "match_events.csv")
print("match_events columns:", list(match_events.columns))
print("match_events event types:", sorted(match_events["event_type"].unique()))
print("-> match_events carries NO xG column; xG is only a per-team-per-match aggregate",
      "(matches.home_xg/away_xg -> team_match_stats.xg).\n")

n_goals = len(goals)
own = goals[goals["owngoal"] == True]
pen = goals[goals["penalty"] == True]
print(f"total goals: {n_goals} | own goals: {len(own)} | penalty goals: {len(pen)}")

# The 14 own goals. `team` is the BENEFITING team; `scorer` is an opposition player.
display(own[["match_id", "team", "scorer", "minute", "penalty", "owngoal"]].reset_index(drop=True))

# Every own-goal match has full xG coverage (so no team is missing an xG total).
og_matches = set(own["match_id"])
print("own-goal matches:", len(og_matches),
      "| with team_stats/xG coverage:", len(og_matches & set(team_stats["match_id"])))

# Concrete cross-check: USA benefited from an own goal vs Paraguay. The OG scorer is a Paraguay
# player, and USA's shots/xG in that match are USA's own attempts — the OG is not among them.
mid = "2026-06-12_USA_vs_Paraguay"
print(f"\ncross-check {mid}:")
display(goals.loc[goals["match_id"] == mid, ["team", "scorer", "minute", "owngoal"]])
display(team_stats.loc[team_stats["match_id"] == mid, ["team", "side", "xg", "shots", "shots_on_target"]])

### Task 0 findings

- **No shot-level xG exists.** xG is only a per-team-per-match aggregate (`team_match_stats.xg`,
  from `matches.home_xg`/`away_xg`). `match_events.csv` has Goal/Assist/Card/VAR events but no xG.
- **An own goal is the opponent's action, credited to the beneficiary.** All 14 own goals are
  attributed to the benefiting team via `goals.team`, but the scorer is an opposition player (e.g.
  USA's own goal vs Paraguay was scored by *Damian Bobadilla*, a Paraguay player). It is never one
  of the benefiting team's shots, so it contributes **0** to that team's aggregate xG.
- **Therefore removal is a clean subtraction:** dropping an own goal removes **1 goal and 0 xG**.
  There is no "dangling" xG left behind, and no sequence turns artificially negative. The two
  residuals differ by exactly the own-goal count:

  > **finishing_residual = scoreboard_residual − own_goals_for**  (xG unchanged) — asserted below.
- **Penalty xG cannot be separated.** xG is only an aggregate team total; there is no shot- or
  penalty-level xG. So the *open-play* finishing residual (non-penalty goals − non-penalty xG) is
  **not computable** from this data and is intentionally omitted in Task 1 rather than guessed.

## Task 1 — The auditable decomposition table

One row per team, sorted by **scoreboard residual**. This table is the source of truth behind every
chart below.

In [ ]:
xg = team_stats.groupby("team")["xg"].sum().rename("xg")
all_goals = goals.groupby("team").size().rename("all_goals")
own_goals_for = own.groupby("team").size().rename("own_goals_for")
penalty_goals = pen.groupby("team").size().rename("penalty_goals")

justice = pd.concat([xg, all_goals, own_goals_for, penalty_goals], axis=1)
justice["all_goals"] = justice["all_goals"].fillna(0).astype(int)
justice["own_goals_for"] = justice["own_goals_for"].fillna(0).astype(int)
justice["penalty_goals"] = justice["penalty_goals"].fillna(0).astype(int)
justice["xg"] = justice["xg"].fillna(0.0)
justice["non_own_goals"] = justice["all_goals"] - justice["own_goals_for"]

justice["scoreboard_residual"] = justice["all_goals"] - justice["xg"]
justice["finishing_residual"] = justice["non_own_goals"] - justice["xg"]

# Task 0 identity: the two residuals differ by exactly own_goals_for (xG never moves).
assert np.allclose(
    justice["finishing_residual"], justice["scoreboard_residual"] - justice["own_goals_for"]
), "finishing_residual must equal scoreboard_residual - own_goals_for"
print("identity holds: finishing_residual == scoreboard_residual - own_goals_for")

# NOTE: an "open-play finishing residual" (non-own, non-penalty goals - non-penalty xG) would need
# penalty xG separated from open-play xG. This dataset only has an aggregate team xG, so that column
# is intentionally omitted (see Task 0) rather than guessed.

justice = justice.sort_values("scoreboard_residual", ascending=False)
cols = ["xg", "all_goals", "own_goals_for", "non_own_goals", "penalty_goals",
        "scoreboard_residual", "finishing_residual"]
display(justice[cols].round(2))

## Task 2 — A concentration metric that doesn't explode

The EDA's `best_match / net_residual` is invalid: when a team has offsetting +/− matches the
denominator collapses toward zero and the ratio explodes (e.g. Canada 6.06). Replace it with two
honest measures, both computed on the **finishing residual** per team-match (own goals already gone):

- **Leave-one-out residual** (headline) = `sum(match_diffs) − max(match_diff)`. Does the team still
  look like an over-performer after dropping its single best match?
- **Positive-contribution concentration** (support) = `max(positive_diff) / sum(positive_diffs)`,
  bounded in [0, 1]: of all a team's good games, how much rides on the best one?

In [ ]:
# Per (match, team) finishing diff: non-own goals in that match minus that team's xG.
non_own_pm = (
    goals[goals["owngoal"] != True].groupby(["match_id", "team"]).size().rename("non_own_goals")
)
mt = team_stats[["match_id", "team", "xg"]].merge(non_own_pm, on=["match_id", "team"], how="left")
mt["non_own_goals"] = mt["non_own_goals"].fillna(0).astype(int)
mt["diff"] = mt["non_own_goals"] - mt["xg"]


def leave_one_out(s: pd.Series) -> float:
    """Total finishing residual after dropping the team's single best match."""
    return float(s.sum() - s.max())


def positive_concentration(s: pd.Series) -> float:
    """Share of a team's positive finishing residual coming from its best match, in [0, 1]."""
    pos = s.clip(lower=0)
    total = pos.sum()
    return float(pos.max() / total) if total > 0 else np.nan


concentration = (
    mt.groupby("team")
    .agg(
        finishing_residual=("diff", "sum"),
        matches=("diff", "size"),
        best_match=("diff", "max"),
        leave_one_out=("diff", leave_one_out),
        pos_concentration=("diff", positive_concentration),
    )
    .sort_values("finishing_residual", ascending=False)
)
display(concentration.round(2))

## Task 3 — Poisson uncertainty funnel

A ~5-match sample makes noise look like skill. Model each team's finishing as
`goals_i | xG_i ~ Poisson(xG_i)` and plot **accumulated xG** (x) against the **goals / xG ratio** (y),
with 80% and 95% reference bands that narrow as xG accumulates. Low-xG teams sit inside wide bands
where deviations are expected; only sustained deviation at high xG is notable.

> **This is a reference model to discourage over-interpretation, not a finishing-talent estimate.**
> It ignores xG calibration error, within-match dependence, real finishing ability, and penalties.
> Do not read any team as "clinical" or "wasteful" from this alone. Ratios use **non-own goals**
> (the finishing quantity), consistent with the rest of the notebook.

In [ ]:
# Team points: accumulated xG vs (non-own goals / xG).
funnel = justice[["xg", "non_own_goals"]].copy()
funnel = funnel[funnel["xg"] > 0]
funnel["ratio"] = funnel["non_own_goals"] / funnel["xg"]

# Poisson reference bands as a function of accumulated xG (lambda). ppf gives goal counts; divide
# by lambda to express as a ratio around the mean of 1.0.
lam = np.linspace(0.5, float(funnel["xg"].max()) * 1.05, 240)
bands = {"95%": (0.025, 0.975, "rgba(120,120,120,0.12)"),
         "80%": (0.10, 0.90, "rgba(120,120,120,0.22)")}

fig = go.Figure()
for label, (lo_q, hi_q, fill) in bands.items():
    lo = poisson.ppf(lo_q, lam) / lam
    hi = poisson.ppf(hi_q, lam) / lam
    fig.add_trace(go.Scatter(x=np.r_[lam, lam[::-1]], y=np.r_[hi, lo[::-1]],
                             fill="toself", fillcolor=fill, line=dict(width=0),
                             hoverinfo="skip", name=f"{label} band"))
fig.add_hline(y=1.0, line=dict(color="black", width=1, dash="dash"))
fig.add_trace(go.Scatter(
    x=funnel["xg"], y=funnel["ratio"], mode="markers+text",
    text=funnel.index, textposition="top center", textfont=dict(size=9),
    marker=dict(size=8, color="#2a9d8f", line=dict(width=0.5, color="white")),
    name="teams",
    hovertemplate="%{text}<br>xG=%{x:.2f}<br>goals/xG=%{y:.2f}<extra></extra>"))
fig.update_layout(
    title=dict(text="Finishing vs Poisson expectation (bands narrow as xG accumulates)"),
    xaxis_title="accumulated xG", yaxis_title="non-own goals / xG",
    template="plotly_white", width=920, height=600, showlegend=True,
    margin=dict(t=60, b=90))
fig.add_annotation(xref="paper", yref="paper", x=0, y=-0.14, showarrow=False,
                   font=dict(size=11, color="gray"), text=STAMP)
fig.show()

## Task 4 — Rank change: scoreboard residual vs finishing residual

The money shot. Each team is ranked twice — by scoreboard residual (all goals − xG) and by finishing
residual (non-own goals − xG). The dumbbell shows how the ranking reshuffles once own-goal fortune is
separated out. Rank 1 = biggest residual.

In [ ]:
ranks = justice.copy()
ranks["rank_scoreboard"] = ranks["scoreboard_residual"].rank(ascending=False, method="min").astype(int)
ranks["rank_finishing"] = ranks["finishing_residual"].rank(ascending=False, method="min").astype(int)
ranks["move"] = ranks["rank_scoreboard"] - ranks["rank_finishing"]  # +ve = climbs when luck removed

# Order top-to-bottom by scoreboard rank so the left column reads 1..N.
ranks = ranks.sort_values("rank_scoreboard", ascending=False)
y = list(range(len(ranks)))

MATERIAL = 3  # annotate teams whose rank shifts by >= this many places
fig = go.Figure()
for yi, (team, r) in zip(y, ranks.iterrows()):
    slid = abs(r["move"]) >= MATERIAL or r["own_goals_for"] > 0
    color = "#e76f51" if r["move"] < 0 else ("#2a9d8f" if r["move"] > 0 else "#adb5bd")
    fig.add_trace(go.Scatter(
        x=[r["rank_scoreboard"], r["rank_finishing"]], y=[yi, yi], mode="lines",
        line=dict(color=color, width=3 if slid else 1),
        opacity=0.9 if slid else 0.35, hoverinfo="skip", showlegend=False))

fig.add_trace(go.Scatter(
    x=ranks["rank_scoreboard"], y=y, mode="markers", name="scoreboard rank",
    marker=dict(size=8, color="#264653"),
    hovertemplate="%{customdata}<br>scoreboard rank=%{x}<extra></extra>",
    customdata=ranks.index))
fig.add_trace(go.Scatter(
    x=ranks["rank_finishing"], y=y, mode="markers", name="finishing rank",
    marker=dict(size=8, color="#e9c46a"),
    hovertemplate="%{customdata}<br>finishing rank=%{x}<extra></extra>",
    customdata=ranks.index))

fig.update_layout(
    title="Rank by scoreboard residual → rank by finishing residual",
    xaxis_title="rank (1 = biggest residual)", template="plotly_white",
    width=880, height=1150, legend=dict(orientation="h", y=1.02, x=0),
    margin=dict(l=140, t=80, b=70))
fig.update_yaxes(tickmode="array", tickvals=y, ticktext=ranks.index, autorange="reversed")
fig.update_xaxes(rangemode="tozero")  # rank 1 sits at the left
fig.add_annotation(xref="paper", yref="paper", x=0, y=-0.05, showarrow=False,
                   font=dict(size=11, color="gray"), text=STAMP)
fig.show()

# The headline mover, stated as fact for the writeup.
usa = ranks.loc["USA"]
print(f"USA: scoreboard residual {usa['scoreboard_residual']:+.2f} (rank {usa['rank_scoreboard']}) "
      f"-> finishing residual {usa['finishing_residual']:+.2f} (rank {usa['rank_finishing']}), "
      f"after {usa['own_goals_for']} opponent own goals.")
display(ranks.loc[ranks["own_goals_for"] > 0,
        ["own_goals_for", "scoreboard_residual", "finishing_residual",
         "rank_scoreboard", "rank_finishing", "move"]].round(2))